In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

BASE_PATH = '/content/drive/My Drive/phishing project datasets/processed'

train_df = pd.read_csv(f'{BASE_PATH}/train_features.csv')
test_df = pd.read_csv(f'{BASE_PATH}/test_features.csv')

for df in [train_df, test_df]:
    df['datetime'] = pd.to_datetime(df['datetime'])

print(train_df.shape, test_df.shape)

Mounted at /content/drive
(202583, 38) (50646, 38)


In [ ]:
enron_recip = pd.read_csv(
    f'{BASE_PATH}/enron_emails_cleaned_temporal.csv',
    usecols=['message_id', 'to']
).rename(columns={'message_id': 'email_id', 'to': 'recipient'})

nazario_raw = pd.read_csv(
    f'{BASE_PATH}/nazario5_cleaned_temporal.csv',
    usecols=['sender_address', 'utc_datetime', 'receiver_address']
)
nazario_raw['utc_datetime'] = pd.to_datetime(nazario_raw['utc_datetime'], errors='coerce')
nazario_raw['sender_norm'] = nazario_raw['sender_address'].astype(str).str.strip().str.lower()
nazario_raw['email_id'] = nazario_raw['sender_norm'] + '_' + nazario_raw['utc_datetime'].astype(str)
nazario_recip = nazario_raw[['email_id', 'receiver_address']].rename(columns={'receiver_address': 'recipient'})

recipients_all = pd.concat([enron_recip, nazario_recip], ignore_index=True).drop_duplicates(subset='email_id')
recipients_all['recipient'] = recipients_all['recipient'].astype(str).str.strip().str.lower()

train_df = train_df.merge(recipients_all, on='email_id', how='left')
test_df = test_df.merge(recipients_all, on='email_id', how='left')

print("Train missing recipient:", train_df['recipient'].isna().sum())
print("Test missing recipient:", test_df['recipient'].isna().sum())

Train missing recipient: 0
Test missing recipient: 0


In [ ]:
print(train_df['recipient'].dropna().sample(10, random_state=1).tolist())

['emca@yahoogroups.com', 'diane.goode@enron.com', 'gloria.ogenyi@enron.com', 'jon.adler@enron.com, alhamd.alkhayat@enron.com, chuck.andrews@enron.com, \n\tn.baker@enron.com, bernie.barcio@enron.com, frank.bay@enron.com, \n\tmichael.bilberry@enron.com, martin.blick@enron.com, \n\ttrey.cash@enron.com, hai.chen@enron.com, elena.chilkina@enron.com, \n\tmark.courtney@enron.com, david.dye@enron.com, \n\tshruti.gandhi-gupta@enron.com, steve.gim@enron.com, \n\tjaime.gualy@enron.com, pearce.hammond@enron.com, \n\tdavid.hensel@enron.com, sarah.hotze@enron.com, \n\trakhi.israni@enron.com, ken.jett@enron.com, michelle.juden@enron.com, \n\tj..kean@enron.com, heather.kendall@enron.com, \n\twilliam.kendrick@enron.com, john.kiani-aslani@enron.com, \n\tkelli.little@enron.com, todd.litton@enron.com, \n\taamir.maniar@enron.com, david.maskell@enron.com, \n\tmaureen.mcvicker@enron.com, tooran.memari@enron.com, \n\tandrew.miles@enron.com, abhijeet.naik@enron.com, tim.nash@enron.com, \n\tlynn.nazareth@enron.

In [ ]:
import re

def parse_recipients(raw):
    if pd.isna(raw) or raw == 'nan':
        return []
    # normalize whitespace/newlines/tabs, then split on commas
    cleaned = re.sub(r'\s+', ' ', str(raw)).strip()
    parts = [p.strip().lower() for p in cleaned.split(',') if p.strip()]
    return parts

train_df['recipient_list'] = train_df['recipient'].apply(parse_recipients)
test_df['recipient_list'] = test_df['recipient'].apply(parse_recipients)

train_df['unique_recipient_count'] = train_df['recipient_list'].apply(len)
test_df['unique_recipient_count'] = test_df['recipient_list'].apply(len)

print(train_df['unique_recipient_count'].describe())

count    202583.000000
mean          4.403923
std          17.259636
min           0.000000
25%           1.000000
50%           1.000000
75%           2.000000
max        1029.000000
Name: unique_recipient_count, dtype: float64


In [ ]:
def build_correspondence_features(df):
    df = df.sort_values(['sender', 'datetime']).reset_index(drop=True)
    n = len(df)
    is_repeat = np.zeros(n, dtype=bool)
    reuse_ratio = np.zeros(n, dtype=np.float32)
    prior_unique_recipients = np.zeros(n, dtype=np.int32)

    for _, idx_group in df.groupby('sender').indices.items():
        idx_group = np.sort(idx_group)
        seen_recipients = set()
        for row_idx in idx_group:
            current_recipients = set(df.at[row_idx, 'recipient_list'])
            overlap = current_recipients & seen_recipients

            is_repeat[row_idx] = len(overlap) > 0
            reuse_ratio[row_idx] = len(overlap) / len(current_recipients) if current_recipients else 0.0
            prior_unique_recipients[row_idx] = len(seen_recipients)

            seen_recipients |= current_recipients  # update AFTER computing this row's features -- keeps it causal

    df['is_repeat_correspondent'] = is_repeat
    df['recipient_reuse_ratio'] = reuse_ratio
    df['sender_prior_unique_recipients'] = prior_unique_recipients
    return df

print("Building train correspondence features...")
train_df = build_correspondence_features(train_df)
print("Building test correspondence features...")
test_df = build_correspondence_features(test_df)

print(train_df.groupby('label')[['is_repeat_correspondent','recipient_reuse_ratio','sender_prior_unique_recipients']].mean())

Building train correspondence features...
Building test correspondence features...
       is_repeat_correspondent  recipient_reuse_ratio  \
label                                                   
0                     0.714274               0.679266   
1                     0.226091               0.226091   

       sender_prior_unique_recipients  
label                                  
0                          114.007288  
1                            3.619547  


In [ ]:
train_df = train_df.drop(columns=['recipient_list'])
test_df = test_df.drop(columns=['recipient_list'])

train_df.to_csv(f'{BASE_PATH}/train_features.csv', index=False)
test_df.to_csv(f'{BASE_PATH}/test_features.csv', index=False)
print("Saved with new correspondence features.")

Saved with new correspondence features.


In [ ]:
BEHAVIORAL_COLS = [
    # domain reputation/age
    'domain_age_days', 'domain_reputation_score', 'tld_risk_weight',
    # auth
    'has_dmarc_record', 'dmarc_enforced', 'spf_pass', 'dkim_pass',
    # IP
    'ip_is_residential_proxy', 'ip_is_vpn_or_anon', 'ip_is_datacenter', 'ip_reputation_score',
    # sender comm history
    'sender_email_count', 'sender_active_days', 'sender_avg_daily_volume',
    # correspondence patterns (new)
    'unique_recipient_count', 'is_repeat_correspondent', 'recipient_reuse_ratio', 'sender_prior_unique_recipients',
    # URL structure
    'has_url', 'is_ip_based', 'subdomain_count', 'path_length', 'query_param_count', 'url_length', 'uses_https',
]

def clean_behavioral(df):
    df = df.copy()
    # 43 rows with missing sender_domain -> missing domain/auth fields; flag + median-fill rather than drop
    domain_cols = ['domain_age_days', 'domain_reputation_score', 'tld_risk_weight']
    df['domain_unknown'] = df['domain_age_days'].isna().astype(float)
    for col in domain_cols:
        df[col] = df[col].fillna(df[col].median())
    for col in ['has_dmarc_record', 'dmarc_enforced', 'spf_pass', 'dkim_pass']:
        df[col] = df[col].fillna(False).astype(float)

    # URL fields: NaN means has_url=0, so 0 is the correct fill, not a missing-data problem
    url_cols = ['is_ip_based', 'subdomain_count', 'path_length', 'query_param_count', 'url_length', 'uses_https']
    for col in url_cols:
        df[col] = df[col].fillna(0).astype(float)

    for col in ['ip_is_residential_proxy', 'ip_is_vpn_or_anon', 'ip_is_datacenter', 'is_repeat_correspondent']:
        df[col] = df[col].astype(float)

    return df

train_df = clean_behavioral(train_df)
test_df = clean_behavioral(test_df)

FINAL_BEHAVIORAL_COLS = BEHAVIORAL_COLS + ['domain_unknown']
print(train_df[FINAL_BEHAVIORAL_COLS].isna().sum().sum(), "remaining NaNs in train")
print(test_df[FINAL_BEHAVIORAL_COLS].isna().sum().sum(), "remaining NaNs in test")

/tmp/ipykernel_1357/4095439954.py:24: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False).astype(float)
/tmp/ipykernel_1357/4095439954.py:24: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False).astype(float)
/tmp/ipykernel_1357/4095439954.py:24: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)

0 remaining NaNs in train
0 remaining NaNs in test


/tmp/ipykernel_1357/4095439954.py:24: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False).astype(float)
/tmp/ipykernel_1357/4095439954.py:24: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False).astype(float)
/tmp/ipykernel_1357/4095439954.py:24: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[FINAL_BEHAVIORAL_COLS].values.astype(np.float32))
X_test = scaler.transform(test_df[FINAL_BEHAVIORAL_COLS].values.astype(np.float32))

y_train = train_df['label'].values
y_test = test_df['label'].values

print(X_train.shape, X_test.shape)

import joblib
joblib.dump(scaler, f'{BASE_PATH}/behavioral_scaler.pkl')

(202583, 26) (50646, 26)


['/content/drive/My Drive/phishing project datasets/processed/behavioral_scaler.pkl']

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class BehavioralDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = BehavioralDataset(X_train, y_train)
test_dataset = BehavioralDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

In [ ]:
class BehavioralFFN(nn.Module):
    def __init__(self, input_dim, hidden_dims=(64, 32)):
        super().__init__()
        self.hidden1 = nn.Sequential(nn.Linear(input_dim, hidden_dims[0]), nn.ReLU(), nn.Dropout(0.2))
        self.hidden2 = nn.Sequential(nn.Linear(hidden_dims[0], hidden_dims[1]), nn.ReLU(), nn.Dropout(0.2))
        self.classifier = nn.Linear(hidden_dims[1], 2)

    def forward(self, x, return_embedding=False):
        h = self.hidden1(x)
        embedding = self.hidden2(h)  # this is what we keep for fusion
        if return_embedding:
            return embedding
        return self.classifier(embedding)

model = BehavioralFFN(input_dim=X_train.shape[1]).to(device)
print(model)

BehavioralFFN(
  (hidden1): Sequential(
    (0): Linear(in_features=26, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
  )
  (hidden2): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
  )
  (classifier): Linear(in_features=32, out_features=2, bias=True)
)


In [ ]:
label_counts = pd.Series(y_train).value_counts().sort_index()
total = label_counts.sum()
class_weights = torch.tensor([total / (2*c) for c in label_counts], dtype=torch.float32).to(device)
print("Class weights:", class_weights)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

NUM_EPOCHS = 10  # cheap model, small input dim -- can afford more passes than the LSTM

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  avg loss: {total_loss/len(train_loader):.4f}")

Class weights: tensor([ 0.5060, 42.4880], device='cuda:0')
Epoch 1/10  avg loss: 0.1841
Epoch 2/10  avg loss: 0.0873
Epoch 3/10  avg loss: 0.0757
Epoch 4/10  avg loss: 0.0698
Epoch 5/10  avg loss: 0.0662
Epoch 6/10  avg loss: 0.0583
Epoch 7/10  avg loss: 0.0562
Epoch 8/10  avg loss: 0.0549
Epoch 9/10  avg loss: 0.0505
Epoch 10/10  avg loss: 0.0483


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

model.eval()
all_probs, all_preds, all_labels = [], [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        logits = model(X_batch)
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds = logits.argmax(dim=1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

print(classification_report(all_labels, all_preds, target_names=['benign','phishing'], digits=4))
print("ROC-AUC:", roc_auc_score(all_labels, all_probs))
print("PR-AUC:", average_precision_score(all_labels, all_probs))

              precision    recall  f1-score   support

      benign     0.9999    0.9737    0.9866     50050
    phishing     0.3101    0.9916    0.4724       596

    accuracy                         0.9739     50646
   macro avg     0.6550    0.9827    0.7295     50646
weighted avg     0.9918    0.9739    0.9806     50646

ROC-AUC: 0.9989533285506439
PR-AUC: 0.9627500588984099


In [ ]:
model.eval()

def extract_embeddings(loader):
    all_embeds = []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            emb = model(X_batch, return_embedding=True)
            all_embeds.append(emb.cpu().numpy())
    return np.vstack(all_embeds)

# use non-shuffled loaders so embedding order matches train_df/test_df row order
train_loader_ordered = DataLoader(train_dataset, batch_size=512, shuffle=False)
test_loader_ordered = DataLoader(test_dataset, batch_size=512, shuffle=False)

train_behavioral_embed = extract_embeddings(train_loader_ordered)
test_behavioral_embed = extract_embeddings(test_loader_ordered)

print(train_behavioral_embed.shape, test_behavioral_embed.shape)
print("NaNs:", np.isnan(train_behavioral_embed).sum(), np.isnan(test_behavioral_embed).sum())

(202583, 32) (50646, 32)
NaNs: 0 0


In [ ]:
EMBED_PATH = f'{BASE_PATH}/embeddings'

np.save(f'{EMBED_PATH}/train_behavioral_embeddings.npy', train_behavioral_embed)
np.save(f'{EMBED_PATH}/test_behavioral_embeddings.npy', test_behavioral_embed)
train_df[['email_id']].to_csv(f'{EMBED_PATH}/train_behavioral_embeddings_index.csv', index=False)
test_df[['email_id']].to_csv(f'{EMBED_PATH}/test_behavioral_embeddings_index.csv', index=False)

torch.save(model.state_dict(), f'{BASE_PATH}/behavioral_ffn_model.pt')

print("Saved behavioral embeddings, index files, model weights, and scaler.")

Saved behavioral embeddings, index files, model weights, and scaler.
